You are given a table that tracks user activities.
Each activity has a start date and end date.

Important conditions:

Activities for a user do not overlap

A user can have one or multiple activities

If a user has only one activity → return that activity

If a user has multiple activities → return the second most recent activity

“Most recent” is determined by endDate

🗂 Table Schema

CREATE TABLE UserActivity (
    username    VARCHAR(20),
    activity    VARCHAR(20),
    startDate   DATE,
    endDate     DATE
);


Expected Output

 
 

In [0]:
%sql
WITH ranked AS (
    SELECT 
        username,
        activity,
        startDate,
        endDate,
        ROW_NUMBER() OVER (PARTITION BY username ORDER BY endDate DESC) AS rn,
        COUNT(*) OVER (PARTITION BY username) AS cnt
    FROM UserActivity
)

SELECT 
    username,
    activity,
    startDate,
    endDate
FROM ranked
WHERE 
    (cnt = 1 AND rn = 1)
    OR
    (cnt > 1 AND rn = 2);

In [0]:
%sql
SELECT 
    ua1.username,
    ua1.activity,
    ua1.startDate,
    ua1.endDate
FROM UserActivity ua1
WHERE 
    (
        -- Case 1: Only one activity
        (SELECT COUNT(*) 
         FROM UserActivity ua2 
         WHERE ua2.username = ua1.username) = 1
    )
    OR
    (
        -- Case 2: Second most recent activity
        (SELECT COUNT(*) 
         FROM UserActivity ua2
         WHERE ua2.username = ua1.username
           AND ua2.endDate > ua1.endDate) = 1
    );

In [0]:
from pyspark.sql import functions as F

# Sample DataFrame: df

# Step 1: Count total activities per user
df_count = df.groupBy("username").agg(F.count("*").alias("cnt"))

# Step 2: Self join to count more recent activities
df_join = df.alias("a").join(
    df.alias("b"),
    (F.col("a.username") == F.col("b.username")) &
    (F.col("b.endDate") > F.col("a.endDate")),
    "left"
)

df_rank = df_join.groupBy(
    "a.username", "a.activity", "a.startDate", "a.endDate"
).agg(
    F.count("b.endDate").alias("recent_count")
)

# Step 3: Join with total count
df_final = df_rank.join(df_count, "username")

# Step 4: Apply filtering logic
result = df_final.filter(
    ((F.col("cnt") == 1) & (F.col("recent_count") == 0)) |
    ((F.col("cnt") > 1) & (F.col("recent_count") == 1))
).select(
    "username", "activity", "startDate", "endDate"
)

result.show()